# 데이터 준비 및 Hugging Face 업로드
이 노트북은 특정 폴더 구조 내의 JSON 파일들을 읽어 Alpaca 포맷으로 변환하고, 이를 Hugging Face Dataset Hub에 업로드하는 과정을 수행합니다.

In [1]:
# 패키지 설치
!pip install pandas datasets huggingface_hub


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import json
import pandas as pd
from datasets import Dataset
from huggingface_hub import notebook_login

def prepareAlpacaDataset(basePath):
    """
    지정된 경로에서 JSON 파일들을 읽어 Alpaca 포맷으로 변환합니다.
    
    Args:
        basePath (str): 데이터셋이 위치한 기본 경로
    
    Returns:
        list: Alpaca 포맷으로 변환된 데이터 리스트
    """
    try:
        alpacaDataList = []
        
        # 모든 하위 디렉토리를 순회
        for root, dirs, files in os.walk(basePath):
            # 폴더 이름에서 과 명칭 추출 (예: TL_내과 -> 내과)
            inputContext = ""
            folderName = os.path.basename(root)
            if "TL_" in folderName:
                inputContext = folderName.split("TL_")[-1]
            
            # 반복문: for i in range(0, len..) 스타일 준수
            for i in range(0, len(files)):
                fileName = files[i]
                if fileName.endswith(".json"):
                    filePath = os.path.join(root, fileName)
                    
                    with open(filePath, "r", encoding="utf-8-sig") as f:
                        jsonData = json.load(f)
                        
                        # Alpaca 포맷 맵핑
                        # instruction: 질문, input: 폴더 기반 정보, output: 답변
                        alpacaItem = {
                            "instruction": jsonData.get("question", ""),
                            "input": inputContext,
                            "output": jsonData.get("answer", "")
                        }
                        alpacaDataList.append(alpacaItem)
        
        return alpacaDataList
    
    except Exception as e:
        print(f"Error during data preparation: {str(e)}")
        return []

def uploadToHuggingFace(dataList, repoId):
    """
    변환된 데이터를 Hugging Face에 업로드합니다.
    
    Args:
        dataList (list): Alpaca 포맷 데이터
        repoId (str): Hugging Face 레포지토리 ID
    """
    try:
        if len(dataList) == 0:
            print("업로드할 데이터가 없습니다.")
            return
            
        # pandas DataFrame을 거쳐 Dataset으로 변환
        df = pd.DataFrame(dataList)
        hfDataset = Dataset.from_pandas(df)
        
        # Hugging Face 업로드
        hfDataset.push_to_hub(repoId)
        print(f"Successfully uploaded to {repoId}")
        
    except Exception as e:
        print(f"Error during upload: {str(e)}")

# 1. 데이터 로드 및 변환
datasetRootPath = "./dataset"
finalAlpacaData = prepareAlpacaDataset(datasetRootPath)

print(f"Total items processed: {len(finalAlpacaData)}")



Total items processed: 15360


In [9]:
# 2. Hugging Face 로그인 (최초 1회 필요)
notebook_login()

# 3. 업로드 실행 (레포지토리 명칭 수정 필요)
uploadToHuggingFace(finalAlpacaData, "yunhwa/MediCore")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Successfully uploaded to yunhwa/MediCore
